Starting with RAG

In [1]:
!pip install langchain langchain-google-genai langchain-core langchain-community pypdf faiss-cpu

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI # helps in configuring the Gemini LLM models
import os # for creating virtual environment
import getpass # allows us to secretly pass our APIs

In [3]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your Gemini API Key")

Enter your Gemini API Key··········


In [4]:
llm=ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [5]:
from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file

/tmp/ipykernel_7572/4271358169.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file


In [6]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")

In [7]:
documents=loader.load()

report=""

for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))

Total number of pages 9


In [8]:
print(report[:1000]) # print first 1000 lines

AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI trainin

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, # inside each chunk how many tokens we have
    chunk_overlap=200 # how many tokens are overlapping between each chunk
)

chunks=text_splitter.split_documents(documents)

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

In [11]:
from langchain_community.vectorstores import FAISS
vector_stores=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [12]:
query="What all investments did AxiOraa make in AI"

In [13]:
# the below process gives results but along with result it also gives too much of data which is not required
# results=vector_stores.similarity_search(query=query)

# for i, doc in enumerate(results):
#  print(f"{i+1}. {doc.page_content}")

#-------------- so we use with score and gauge with confidence -----------#
results=vector_stores.similarity_search_with_score(query=query)
for i, (doc,score) in enumerate(results):
 print(f"RESULT{i}")
 print(f"Score:{score}")
 print(f"Document:{doc.page_content[:500]}") # here 500 denotes the number of token to be used at max

RESULT0
Score:0.5484022498130798
Document:AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrati
RESULT1
Score:0.5836595296859741
Document:engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insur

In [14]:
retriever=vector_stores.as_retriever(
    search_type="similarity", #performing similarity between vector embeddings and query embeddings
    search_kwargs={"k":1} # generating top 1 response
)

In [15]:
from langchain_core.prompts import ChatPromptTemplate

In [17]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior Management Consultant and your job is to answer the user questions based on the retrieved context.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved context:{context}
    Question:{question}

    Provide:
    1. Answer
    2. Supporting Evidence
    """
)

In [18]:
from langchain_core.output_parsers import StrOutputParser

In [19]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [22]:
# question="Talk about the business overview of this company"
question="help me understand the financial and operational highlights of this company"
retrieved_docs=retriever.invoke(question)

context="\n\n".join([doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question,
})

In [23]:
print(response)

**1. Answer**

Based on the provided context, the financial and operational highlights of the company are as follows:

* **Revenue Growth:** Revenue grew by 21.1%, increasing from USD 2.18 billion to USD 2.64 billion.
* **Gross Margin Expansion:** Gross Margin improved from 41.5% to 44.2%.
* **EBITDA Margin:** EBITDA Margin reached 22.8%.
* **Net Profit Increase:** Net Profit increased to USD 352 million.
* **Cash Flow Performance:** Free Cash Flow remained strong.
* **Operational Insights:** The section includes narrative analysis and management commentary providing further operational context.

---

**2. Supporting Evidence**

* *"Revenue grew from USD 2.18B to USD 2.64B (21.1%)."*
* *"Gross Margin improved from 41.5% to 44.2%."*
* *"EBITDA Margin reached 22.8%."*
* *"Net Profit increased to USD 352M."*
* *"Free Cash Flow remained strong."*
* *"This section includes narrative analysis, management commentary, illustrative"*
